In [ ]:
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import  accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold
import json

In [7]:
def load_dta():
    dataset = load_dataset("pubmed_qa", "pqa_labeled")
    model_namee = 'NeuML/pubmedbert-base-embeddings'
    encder = SentenceTransformer(model_namee)
    questions = [row ['question'] for row in dataset['train']]
    contexts = [row['context'] ['contexts'][-1] for row in dataset ['train']]
    labels = [row['final_decision'] for row in dataset['train']]
    Q = encder.encode(questions) 
    C = encder.encode(contexts)  
    return dataset, Q, C, labels 

dataset, questions, context, labels = load_dta()

In [ ]:
with open('test_ground_truth.json',  'r') as f:
    ground_truth = json.load(f)
testids = set(ground_truth.keys())

ids = [row['pubid'] for row in dataset['train']]
dev = [i for i, pid in enumerate(ids) if str(pid) not in testids]
testt = [i for i, pid in enumerate(ids) if str(pid) in testids]

def buildIS(Q, C):
    abs_diff = Q - C
    elementwise_prod = Q * C
    return np.hstack([abs_diff, elementwise_prod])

X = buildIS(questions, context) 
y = np.array(labels)

X_dev, y_dev = X[dev], y[dev]
X_test, y_test = X[testt], y[testt]

In [9]:
def train_lda():
    skf = StratifiedKFold(n_splits = 10, shuffle = True, random_state = 42)
    accuraccies = []
    f1s = []
    sclrs = []
    mdls = []
    for train, val in skf.split(X_dev, y_dev):
        X_train, X_val = X[train], X[val]
        y_train, y_val = y[train], y[val]
        sclr = StandardScaler()
        train_scaled = sclr.fit_transform(X_train)
        val_scaled = sclr.transform(X_val)
        lda = LinearDiscriminantAnalysis()
        lda.fit(train_scaled, y_train)
        predictds = lda.predict(val_scaled)
        accuraccies.append(accuracy_score(y_val, predictds))
        f1s.append(f1_score(y_val, predictds, average = 'macro'))
        sclrs.append(sclr)
        mdls.append(lda)
    return sclrs, mdls

sclrs, mdls = train_lda()

In [10]:
probs = []
for scaler, classifier in zip(sclrs, mdls):
    test_scaled = scaler.transform(X_test)
    probs2 = classifier.predict_proba(test_scaled) 
    probs.append(probs2)

class_names = mdls[0].classes_
predictons = np.argmax(np.mean(probs, axis = 0), axis = 1)
strngs = [class_names[p] for p in predictons]
print(f"Accuracy: {accuracy_score(y_test, strngs):.2f}")
print(f"Macro F1: {f1_score(y_test, strngs, average='macro'):.2f}")

Accuracy: 0.66
Macro F1: 0.58
